In [2]:
import pandas as pd
import numpy as np
import scipy.stats as ss
import math as mt
import itertools

In [3]:
data = pd.read_csv("D:\\graduate\\AB test\\ab_data.csv")
df = data.copy()
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [4]:
df.info()
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB
(294478, 5)


# data cleaning

In [5]:
df['user_id'].nunique()

290584

In [6]:
# make sure treatmennt matches with new_page and control matches with old_page
i = df[((df['group'] == 'treatment') == (df['landing_page'] == 'new_page')) == False].index
df2 = df.drop(i)

In [7]:
df2.shape

(290585, 5)

In [8]:
df2[df2.duplicated(['user_id'], keep=False)]

,user_id,timestamp,group,landing_page,converted
1899,773192,2017-01-09 05:37:58.781806,treatment,new_page,0
2893,773192,2017-01-14 02:55:59.590927,treatment,new_page,0


In [9]:
df2.drop_duplicates(subset = 'user_id', keep = 'first',inplace = True)

In [10]:
# probability of conversion
P_pool = (df2.query('converted == 1').shape[0]) / df2.shape[0]
P_pool

0.11959708724499628

In [11]:
control_df = df2.query("group == 'control'")
P_old = control_df['converted'].mean()
P_old

np.float64(0.1203863045004612)

In [12]:
treatment_df = df2.query("group == 'treatment'")
P_new = treatment_df['converted'].mean()
P_new

np.float64(0.11880806551510564)

# Metrics

In [13]:
N_new = df2.query("landing_page == 'new_page'").landing_page.count()
N_old = df2.query("landing_page == 'old_page'").landing_page.count()
proportion = (N_old/df2.shape[0], N_new/df2.shape[0])
proportion

(np.float64(0.4999380557773312), np.float64(0.5000619442226688))

In [ ]:
# input 1-alpha/2
def get_z_score(alpha):
    return ss.norm.ppf(alpha)

In [25]:
# guardrail cheak 
sd = round(mt.sqrt((0.5*(1-0.5))/df2.shape[0]),4)
CI = (0.5 - sd*get_z_score(1-0.05/2),0.5+sd*get_z_score(1-0.05/2))
CI

(np.float64(0.49823603241391395), np.float64(0.5017639675860861))

In [26]:
N_old/ df2.shape[0]

np.float64(0.4999380557773312)

In [19]:
# calculate minmium sample size
n = 16*(0.12*(1-0.12))/(0.0035**2)
n

137926.53061224488

In [18]:
CI_old = (P_old - get_z_score(1-0.05/2)*mt.sqrt(P_old*(1-P_old)/N_old),P_old + get_z_score(1-0.05/2)*mt.sqrt(P_old*(1-P_old)/N_old))
CI_new = (P_new - get_z_score(1-0.05/2)*mt.sqrt(P_new*(1-P_new)/N_new),P_new + get_z_score(1-0.05/2)*mt.sqrt(P_new*(1-P_new)/N_new))
CI_old, CI_new

((np.float64(0.11871294722381814), np.float64(0.12205966177710426)),
 (np.float64(0.11714442856134422), np.float64(0.12047170246886707)))

CI_old overlap CI_new, This means that the new page is not better than the old page.

In [20]:
# Z-TEST
import statsmodels.api as sm

In [21]:
convert_old = df2.query("landing_page == 'old_page' and converted == 1").shape[0]
convert_new = df2.query("landing_page == 'new_page' and converted == 1").shape[0]

In [22]:
z_score, p_value = sm.stats.proportions_ztest([convert_old, convert_new], [N_old, N_new], alternative='smaller')
z_score, p_value

(np.float64(1.3109241984234394), np.float64(0.9050583127590245))

In [ ]:
# F-TEST robustness check

SE_new = mt.sqrt(P_new*(1-P_new)/N_new)
SE_old = mt.sqrt(P_old*(1-P_old)/N_old)
p = 1 - ss.f.cdf(pow(SE_new,2)/pow(SE_old,2), N_new - 1, N_old - 1)
p

np.float64(0.986811599380792)

 variances are the same

In [28]:
# Pooled Confidence Interval
#how much is changed
# is this change statistically significant
# is this change worth it economically

In [29]:
d_hat = P_new - P_old
SE_pool = mt.sqrt(P_pool*(1-P_pool)*(1/N_old+1/N_new))
CI_diff = (d_hat - get_z_score(1-0.05/2)*SE_pool, d_hat + get_z_score(1-0.05/2)*SE_pool)

In [31]:
d_hat,CI_diff

(np.float64(-0.0015782389853555567),
 (np.float64(-0.003937865555689694), np.float64(0.0007813875849785809)))

In [32]:
# Chi-Squared Test:

In [33]:
treatment_converted = treatment_df.converted.sum()
treatment_not_converted = treatment_df.size - treatment_df.converted.sum()
control_converted = control_df.converted.sum()
control_not_converted = control_df.size - control_df.converted.sum()

#create the array to do our chi-squared test: treatment/control along the rows and converted/not converted along the columns:
Chi = np.array([[treatment_converted,treatment_not_converted],[control_converted,control_not_converted]])
Chi

array([[ 17264, 709286],
       [ 17489, 708881]])

In [34]:
print(ss.chi2_contingency(Chi,correction=False)[1])

0.2131252933770616
